<a href="https://colab.research.google.com/github/gibranfp/CursoAprendizajeAutomatizado/blob/2026-2/notebooks/4a_redes_bayesianas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Representación e inferencia en redes bayesianas mediante `pgmpy`
En esta libreta definiremos una red bayesiana simple usando la biblioteca `pgmpy` y haremos inferencia mediante el método de eliminación de variables.

## Instalación
Primero instalamos la biblioteca:

In [1]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 13.2 MB/s eta 0:00:00


## Representación
Considera la siguiente red bayesiana con variables binarias:

![https://raw.githubusercontent.com/gibranfp/CursoAprendizajeAutomatizado/refs/heads/master/fig/rb_entrevista.svg](https://raw.githubusercontent.com/gibranfp/CursoAprendizajeAutomatizado/refs/heads/master/fig/rb_entrevista.svg)

El módulo `models` cuenta con la clase `DiscreteBayesianNetwork` para poder crearla. La clase recibe como argumento una lista de pares (`tuple`) que representan las aristas del grafo dirigido:

In [2]:
from pgmpy.models import DiscreteBayesianNetwork

modelo = DiscreteBayesianNetwork([('Experiencia', 'Entrevista'),
                                  ('Promedio', 'Entrevista'),
                                  ('Entrevista', 'Oferta'),
                                  ('Promedio', 'Posgrado')])

Definimos la tabla de probabilidad condicional (CPD, por sus siglas en inglés) de cada nod usando la clase `TabularCPD` del submódulo `discrete` del módulo `factors`, la cual recibe como argumento el nombre del nodo (`variable=nombre`, donde `nombre` debe coincidir con alguna cadena de la lista de pares), el número de parámetros de la variable (`variable_card=numero`), las probabilidades de todas las combinaciones como lista de listas (`values=lista_de_listas`), los nodos padres (`evidence=lista_nombres`) y el número de parámetros de cada padre (`evidence_card=lista_de_numeros`).

In [3]:
from pgmpy.factors.discrete import TabularCPD

cpd_exp = TabularCPD(variable = 'Experiencia', variable_card = 2,
                     values = [[0.7], [0.3]])
cpd_prom = TabularCPD(variable = 'Promedio', variable_card = 2,
                      values = [[0.2], [0.8]])
cpd_ent = TabularCPD(variable = 'Entrevista', variable_card = 2,
                     values = [[0.9, 0.6, 0.3, 0.1],
                               [0.1, 0.4, 0.7, 0.9]],
                     evidence = ['Experiencia', 'Promedio'],
                     evidence_card = [2, 2])
cpd_ofe = TabularCPD(variable = 'Oferta', variable_card = 2,
                     values = [[0.9, 0.2], [0.1, 0.8]],
                     evidence = ['Entrevista'],
                     evidence_card = [2])
cpd_pos = TabularCPD(variable = 'Posgrado', variable_card = 2,
                     values = [[0.9, 0.2], [0.1, 0.8]],
                     evidence = ['Promedio'],
                     evidence_card = [2])

Asociamos las tablas a la red bayesiana que creamos usando el método `add_cpds` de la instancia de `DiscreteBayesianNetwork`:

In [4]:
modelo.add_cpds(cpd_exp, cpd_prom, cpd_ent, cpd_pos, cpd_ofe)

Verificamos que la red bayesiana sea correcta con el método `check_model`:

In [5]:
modelo.check_model()

True

## Independencia condicional
Probamos algunas propiedades de independencia condicional:

In [6]:
print(modelo.is_dconnected('Experiencia', 'Promedio'))
print(modelo.is_dconnected('Experiencia', 'Promedio', observed=['Entrevista']))
print(modelo.is_dconnected('Entrevista', 'Posgrado'))
print(modelo.is_dconnected('Entrevista', 'Posgrado', observed=['Promedio']))


False
True
True
False


Obtenemos la cobija de Markov para algunas variables:

In [7]:
modelo.get_markov_blanket('Experiencia'), modelo.get_markov_blanket('Promedio')

(['Promedio', 'Entrevista'], ['Posgrado', 'Experiencia', 'Entrevista'])

Ahora las relaciones de independencia a partir de su cobija de Markov:

In [8]:
modelo.local_independencies('Experiencia'), modelo.local_independencies('Promedio')

((Experiencia ⟂ Posgrado, Promedio), (Promedio ⟂ Experiencia))

Para obtener las relaciones de independencia de todas las variables del grafo por separación D:

In [9]:
modelo.get_independencies().independencies

[(Oferta ⟂ Promedio | Entrevista),
 (Experiencia ⟂ Oferta | Entrevista),
 (Entrevista ⟂ Posgrado | Promedio),
 (Experiencia ⟂ Promedio),
 (Oferta ⟂ Posgrado | Entrevista),
 (Experiencia ⟂ Posgrado)]

## Inferencia via eliminación de variables
Deseamos inferir la probabilidad de experiencia dado que hubo una oferta de trabajo:

$$
P(E \mid O = 1) = \frac{P(E, O=1)}{P(O=1)}
$$

Para esta inferencia vamos a utilizar el método de eliminación de variables. En ``pgmpy` se encuentra implementado en la clase `VariableElimination`, cuyo constructor recibe como argumento el modelo (red bayesiana creada). Posteriormente se realiza la consulta usando el método `query` de la instancia de `VariableElimination`, que recibe como argumento las variables de consulta (`variables=lista_de_nombres`) y las evidencias (`evidence=diccionario_nombre:valor`).

In [10]:
from pgmpy.inference import VariableElimination
exp_inf = VariableElimination(modelo)

# haciendo consulta P(E | O = 1)
consulta_exp_ofe = exp_inf.query(variables = ['Experiencia'],
                                 evidence = {'Oferta': 1})
print(consulta_exp_ofe)

/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in a future release. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


+----------------+--------------------+
| Experiencia    |   phi(Experiencia) |
+================+====================+
| Experiencia(0) |             0.5291 |
+----------------+--------------------+
| Experiencia(1) |             0.4709 |
+----------------+--------------------+
